In [ ]:
import sys
from pathlib import Path

# Get the root directory of the project
project_root = Path("../../KCFM_tutorial/cell_atlases").parent.parent
# project_root = Path(__file__).parent.parent
sys.path.append(str(project_root))
import os
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'

In [ ]:
import os
import pickle
import torch
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from models.gene_tokenizer import GeneVocab
from models.model import MambaModel

In [ ]:
# configuration
max_seq_len = 2049
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
class SingleParquetFileDataset(Dataset):
    """Read the dataset of a single parquet file"""
    def __init__(self, file_path):
        self.df = pq.read_table(file_path).to_pandas()
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        return {
            'X': torch.FloatTensor(row['X']),
            'cell_type': row['cell_type']
        }

In [ ]:
def preprocess_batch(x_batch, feature_names, vocab, device):
    """Preprocess batch data into model input format"""
    batch_size = x_batch.size(0)
    values_tensor = torch.full((batch_size, max_seq_len), -2.0, device=device)
    src_tensor = torch.full((batch_size, max_seq_len), vocab["<pad>"], dtype=torch.long, device=device)
    padding_mask = torch.zeros((batch_size, max_seq_len), dtype=torch.bool, device=device)
    
    nonzero_mask = x_batch > 0
    nonzero_counts = nonzero_mask.sum(dim=1)
    
    for i in range(batch_size):
        nz_count = min(nonzero_counts[i], max_seq_len-1)
        nz_idx = torch.nonzero(x_batch[i])[:nz_count].squeeze()
        nz_values = x_batch[i][nz_idx]
        
        values_tensor[i, 0] = 0.0
        values_tensor[i, 1:1+nz_count] = nz_values
        
        genes = ["<cls>"] + [feature_names[idx] for idx in nz_idx.cpu().numpy()]
        genes += ["<pad>"] * (max_seq_len - len(genes))
        
        src_tensor[i] = torch.tensor([
            vocab[gene] if gene in vocab else vocab["<unk>"] for gene in genes
        ], device=device)
        
        padding_mask[i] = torch.tensor([gene == "<pad>" for gene in genes], device=device)
    
    return values_tensor, src_tensor, padding_mask

In [ ]:
def evaluate_single_file(model, file_path, feature_names, vocab):
    """Evaluate individual files and return predicted results"""
    dataset = SingleParquetFileDataset(file_path)
    loader = DataLoader(
        dataset,
        batch_size=256,
        shuffle=False,
        num_workers=4,
        pin_memory=True
    )
    
    preds = []
    with torch.no_grad():
        for batch in tqdm(loader, desc=f"Processing {os.path.basename(file_path)}"):
            x = batch['X'].to(device, non_blocking=True)
            
            values, src, mask = preprocess_batch(x, feature_names, vocab, device)
            
            with torch.cuda.amp.autocast(enabled=True):
                outputs = model(src=src, values=values, src_key_padding_mask=mask)
            
            preds.append(outputs["cls_output"].argmax(dim=1).cpu().numpy())
    
    return np.concatenate(preds)  # return a 1 x n prediction array

In [ ]:
def load_model(model_path, vocab_path):
    """Load pre trained model and vocabulary list"""
    vocab = GeneVocab.from_file(vocab_path)
    model = MambaModel(
        ntoken=len(vocab), d_model=512,nlayers=6,
        dropout=0.2, pad_token="<pad>", pad_value=-2,
        input_emb_style="continuous", cell_emb_style="cls"
    )
    # print(model.state_dict().keys()) 
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.to(device)
    model.eval()
    return model, vocab

In [ ]:
# 1. Load dataset
df = pd.read_parquet('../../data/merlin_cxg_2023_05_15_sf-log1p/var.parquet')
feature_names = df['feature_name'].tolist()

# 2. Load model
model, vocab = load_model(
    "../../model_state/cell_cls_2loss_check.pth",
    "../../models/vocab.json"
)

# 3. Retrieve all test files (sorted by number)
test_dir = "../../data/merlin_cxg_2023_05_15_sf-log1p/test"
# test_dir = "../../data/merlin_cxg_2023_05_15_sf-log1p/test"
test_files = sorted(
    [os.path.join(test_dir, f) for f in os.listdir(test_dir) if f.endswith('.parquet')],
    key=lambda x: int(os.path.basename(x).split('.')[1])
)

# 4. Generate all predicted results
all_preds = np.array([], dtype=np.int64)
for file_path in tqdm(test_files, desc="Processing all files"):
    preds = evaluate_single_file(model, file_path, feature_names, vocab)
    all_preds = np.concatenate([all_preds, preds])

# 5. Save as pkl file (1 x n format)
output_path = "../../PKL/mamba_predictions_2loss_6layer.pkl"
with open(output_path, 'wb') as f:
    pickle.dump(all_preds, f)

print(f"\nThe predicted results have been saved to {output_path}")
print(f"Total number of predicted samples: {len(all_preds)}")
print(f"Shape of predicted results: {all_preds.shape}")  # Should output (n,)

In [ ]:
# 1. Load dataset
df = pd.read_parquet('../../data/merlin_cxg_2023_05_15_sf-log1p/var.parquet')
feature_names = df['feature_name'].tolist()

# 2. Load model
model, vocab = load_model(
    "../../model_state/cell_cls_3loss_6layer_final.pth ",
    "../../models/vocab.json"
)

# 3. Retrieve all test files (sorted by number)
test_dir = "../../data/merlin_cxg_2023_05_15_sf-log1p/test"
# test_dir = "../../data/merlin_cxg_2023_05_15_sf-log1p/test"
test_files = sorted(
    [os.path.join(test_dir, f) for f in os.listdir(test_dir) if f.endswith('.parquet')],
    key=lambda x: int(os.path.basename(x).split('.')[1])
)

# 4. Generate all predicted results
all_preds = np.array([], dtype=np.int64)
for file_path in tqdm(test_files, desc="Processing all files"):
    preds = evaluate_single_file(model, file_path, feature_names, vocab)
    all_preds = np.concatenate([all_preds, preds])

# 5. Save as pkl file (1 x n format)
output_path = "../../PKL/mamba_predictions_3loss_6layer.pkl"
with open(output_path, 'wb') as f:
    pickle.dump(all_preds, f)

print(f"\nThe predicted results have been saved to {output_path}")
print(f"Total number of predicted samples: {len(all_preds)}")
print(f"Shape of predicted results: {all_preds.shape}")  # Should output (n,)